# Capitulo 15 - RAG com Agentes (Agentic RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap15_rag_agentes.ipynb)

Neste capitulo, construimos agentes inteligentes que utilizam RAG como ferramenta.
Usamos LangGraph para orquestrar o fluxo de decisao do agente.

---

## Atribuicao e Fonte Original

**Fonte:** [RAG-with-Python-Cookbook](https://github.com/PacktPublishing/RAG-with-Python-Cookbook) - Packt Publishing

**Notebooks fonte:** ch08_agentic_rag/8.8_agentic_system_langgraph/building_agents_using_langgraph.ipynb

**Adaptado e comentado por Allan Eric Jepsen** para o livro *LLM On-Premise: Guia Pratico*.

Todos os creditos aos autores originais sao mantidos.

---

## Instalacao de Dependencias

Execute a celula abaixo para instalar todos os pacotes necessarios.

In [ ]:
# Instalar dependencias do capitulo
%pip install -q openai chromadb google-generativeai langchain-core \
    langchain-openai langgraph pydantic python-dotenv

## Introducao ao Agentic RAG

RAG Agentico combina a capacidade de recuperacao de informacao com agentes autonomos.
O agente decide **quando** e **como** buscar informacao, ao inves de seguir um pipeline fixo.

**LangGraph** e um framework que permite construir agentes como grafos de estados,
onde cada no executa uma acao e as transicoes sao determinadas pela logica do agente.

### Building Agents using LangGraph

In [ ]:
%pip install openai chromadb google-generativeai langchain-core langchain-openai langgraph pydantic geopy requests ipython python-dotenv

**LangGraph:** Construimos o grafo de estados do agente.

In [ ]:
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage
# helper function to add messages to the state
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    """The state of the agent."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    number_of_steps: int


In [ ]:
from langchain_core.tools import tool
from geopy.geocoders import Nominatim
from pydantic import BaseModel, Field
import requests

geolocator = Nominatim(user_agent="weather-app")

class SearchInput(BaseModel):
    location: str = Field(
        description="The city and state, e.g., San Francisco"
    )
    date: str = Field(
        description="the forecasting date for when to get "
        "the weather format (yyyy-mm-dd)"
    )

@tool("get_weather_forecast", args_schema=SearchInput, return_direct=True)
def get_weather_forecast(location, date):
    """Retrieves the weather using Open-Meteo API for a given
    location (city) and a date (yyyy-mm-dd).
    Returns a dict mapping time -> temperature for each hour.
    """
    location = geolocator.geocode(location)
    if location:
        try:
            url = "https://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": location.latitude,
                "longitude": location.longitude,
                "hourly": "temperature_2m",
                "start_date": date,
                "end_date": date,
            }
            response = requests.get(url, params=params)
            data = response.json()
            hourly = data.get("hourly", {})
            times = hourly.get("time", [])
            temps = hourly.get("temperature_2m", [])
            result = {}
            for t, temp in zip(times, temps):
                result[t] = temp
            return result
        except Exception as e:
            return {"error": str(e)}
    else:
        return {"error": "Location not found"}

class AudioInput(BaseModel):
    audio_path: str = Field(
        description="Path to the audio file to transcribe"
    )

@tool("transcribe_audio", args_schema=AudioInput, return_direct=True)
def transcribe_audio(audio_path):
    """Transcribe audio file into text using OpenAI Whisper."""
    try:
        with open(audio_path, "rb") as audio_file:
            transcript = openai.Audio.transcribe(
                model="whisper-1",
                file=audio_file
            )
            return transcript.text
    except Exception as e:
        return {"error": str(e)}

tools = [get_weather_forecast, transcribe_audio]


**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
from datetime import datetime
from langchain_openai import ChatOpenAI

# Criar LLM class
llm = ChatOpenAI(
    model="gpt-5.2",
    temperature=1.0,
    max_retries=2,
)

# Bind tools to the model
model = llm.bind_tools([get_weather_forecast])

# Testar the model with tools
query = f"What is the weather in Berlin on {datetime.today()}?"
try:
    res = model.invoke(query)
except Exception as exc:
    res = {"error": str(exc)}

print(res)


In [ ]:
from langchain_core.messages import ToolMessage, AIMessage
from langchain_core.runnables import RunnableConfig

tools_by_name = {}
for _t in tools:
    tools_by_name[_t.name] = _t

# Definir our tool node
def call_tool(state: AgentState):
    outputs = []
    # Iterate over the tool calls in the last message
    for tool_call in state["messages"][-1].tool_calls:
        # Obter the tool by name and arguments in smaller steps
        # to keep lines short
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_obj = tools_by_name[tool_name]
        tool_result = tool_obj.invoke(tool_args)
        outputs.append(
            ToolMessage(
                content=tool_result,
                name=tool_name,
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}

def call_model(
    state: AgentState,
    config: RunnableConfig,
):
    # Invoke the model with the system prompt and the messages
    try:
        response = model.invoke(state["messages"], config)
    except Exception as exc:
        response = AIMessage(content=f"LLM unavailable during graph run: {exc}")
    # Return a list, because this will get added to the
    # existing messages state using the add_messages reducer
    return {"messages": [response]}


# Definir the conditional edge that determines whether to continue or not
def should_continue(state: AgentState):
    messages = state["messages"]
    # If the last message is not a tool call, then finish
    if not getattr(messages[-1], "tool_calls", None):
        return "end"
    # default to continue
    return "continue"


**LangGraph:** Construimos o grafo de estados do agente.

In [ ]:
from langgraph.graph import StateGraph, END

# Definir a new graph with our state
workflow = StateGraph(AgentState)

# 1. Add our nodes
workflow.add_node("llm", call_model)
workflow.add_node("tools",  call_tool)
# 2. Set the entrypoint as `agent`, this is the first node called
workflow.set_entry_point("llm")
# 3. Add a conditional edge after the `llm` node is called.
workflow.add_conditional_edges(
    # Edge is used after the `llm` node is called.
    "llm",
    # The function that will determine which node is called next.
    should_continue,
    # Mapping for where to go next, keys are strings from the
    # function return, and the values are other nodes.
    # END is a special node marking that the graph is finish.
    {
        # If `tools`, then call the tool node.
        "continue": "tools",
        # Otherwise finish.
        "end": END,
    },
)
# 4. Add a normal edge after `tools` is called, `llm` node is called next.
workflow.add_edge("tools", "llm")

# Agora you can compile and visualize the graph
graph = workflow.compile()


**Importacoes:** Importamos as bibliotecas necessarias.

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
from datetime import datetime

# Criar our initial message dictionary
inputs = {
    "messages": [
        ("user", f"What is the weather in Berlin on {datetime.today()}?")
    ]
}

# Call our graph with streaming to see the steps
for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()


In [ ]:
state["messages"].append(("user", "Would it be in Munich warmer?"))

for state in graph.stream(state, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()


In [ ]:
from datetime import datetime

# Criar our initial message dictionary
inputs = {"messages": [("user", f"What is the weather in Berlin on {datetime.today()}?")]}

# Call our graph with streaming to see the steps
for state in graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()

In [ ]:
state["messages"].append(("user", "Would it be in Munich warmer?"))

for state in graph.stream(state, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()